# 02 — Auxotrophy benchmark reproduction

**Workflow version:** 0.5.2

Reproduce the 147 gene-compound phenotype benchmark using COBRApy. The reference run uses a pristine model loaded once from SBML and creates an independent model copy for every record. This preserves pair isolation without repeatedly reparsing the large SBML file. Progress output is append-only so it remains visible in VS Code notebooks.


In [ ]:
from pathlib import Path
import gc
import hashlib
import json
import math
import re
import time

import numpy as np
import pandas as pd
import cobra
from cobra.io import read_sbml_model

SOLVER = "glpk"
UPTAKE_LOWER_BOUND = -1000.0
VIABILITY_FRACTION = 0.01
ISOLATION_MODE = "pristine_copy"

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd
DATA_DIR = ROOT / "data" / "raw"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

ORIGINAL_XML = DATA_DIR / "yeast9.0.xml"
CURATED_XML = DATA_DIR / "Yeast9_curated.xml"
DATASET_XLSX = DATA_DIR / "mmc3.xlsx"

ALIASES = {
    "Yeast9": {},
    "Yeast9_curated": {"a_0001": "r_temp1"},
}

In [ ]:
REQUIRED_COLUMNS = [
    "Gene Systematic Name", "Chemical", "exchange", "ID",
    "Strain Background", "Reference",
]

def split_plus(value):
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split("+") if part.strip()]

def split_genes(value):
    if pd.isna(value):
        return []
    return [
        part.strip()
        for part in re.split(r"\s+and\s+", str(value).strip(), flags=re.I)
        if part.strip()
    ]

def count_exchange_targets(value):
    if pd.isna(value):
        return 0
    text = str(value).strip()
    return len(re.findall(r"\bexchange\b", text, flags=re.I))

def parse_dataset(path: Path) -> pd.DataFrame:
    raw = pd.read_excel(path, sheet_name="all")
    missing = [column for column in REQUIRED_COLUMNS if column not in raw.columns]
    if missing:
        raise ValueError(f"Missing Dataset 2 columns: {missing}")

    records = []
    for idx, row in raw.iterrows():
        excel_row = idx + 2
        genes = split_genes(row["Gene Systematic Name"])
        ids = split_plus(row["ID"])
        chemical = str(row["Chemical"]).strip()
        n_targets = count_exchange_targets(row["exchange"])
        conditional = bool(re.search(r"\badd\b", chemical, flags=re.I))

        if not genes or not ids or n_targets < 1 or len(ids) < n_targets:
            raise ValueError(f"Cannot parse Dataset 2 Excel row {excel_row}.")

        rescue_ids = ids[:n_targets] if conditional else ids
        background_ids = ids[n_targets:] if conditional else []

        records.append({
            "pair_id": f"excel:{excel_row}",
            "excel_row": excel_row,
            "gene_field": str(row["Gene Systematic Name"]).strip(),
            "genes": genes,
            "n_genes": len(genes),
            "chemical": chemical,
            "exchange_field": str(row["exchange"]).strip(),
            "id_field": str(row["ID"]).strip(),
            "rescue_ids": rescue_ids,
            "background_ids": background_ids,
            "conditional_medium": conditional,
            "strain_background": str(row["Strain Background"]).strip(),
            "reference": str(row["Reference"]).strip(),
        })

    pairs = pd.DataFrame(records)
    if len(pairs) != 147:
        raise AssertionError(f"Expected 147 records, found {len(pairs)}.")
    if pairs["pair_id"].duplicated().any():
        raise AssertionError("pair_id values are not unique.")
    return pairs

pairs = parse_dataset(DATASET_XLSX)
display(pairs.head())
print("Records:", len(pairs))
print("Conditional-medium records:", int(pairs["conditional_medium"].sum()))
print("Multi-gene records:", int((pairs["n_genes"] > 1).sum()))

In [ ]:
def solve_growth(model):
    value = model.slim_optimize(error_value=np.nan)
    return str(model.solver.status).lower(), float(value) if value is not None else np.nan

def resolve_reaction_id(model, reaction_id, aliases):
    if reaction_id in model.reactions:
        return reaction_id
    alias = aliases.get(reaction_id)
    if alias is not None and alias in model.reactions:
        return alias
    return None

def classify_auxotrophy(ko_status, ko_growth, rescue_status, rescue_growth, threshold):
    if ko_status != "optimal" or rescue_status != "optimal":
        return "solver_error"
    if not np.isfinite(ko_growth) or not np.isfinite(rescue_growth):
        return "solver_error"
    if ko_growth >= threshold:
        return "type_I"
    if rescue_growth < threshold:
        return "type_II"
    return "correct"

def simulate_pair(model, record, threshold, aliases):
    missing_genes = [gene for gene in record["genes"] if gene not in model.genes]
    if missing_genes:
        return {
            "pair_id": record["pair_id"], "excel_row": record["excel_row"],
            "gene_field": record["gene_field"], "chemical": record["chemical"],
            "classification": "input_error", "correct": False,
            "ko_status": "input_error", "ko_growth": np.nan,
            "rescue_status": "input_error", "rescue_growth": np.nan,
            "missing_genes": "+".join(missing_genes),
            "missing_background": "", "missing_rescue": "",
            "mapped_background": "", "mapped_rescue": "",
        }

    for gene in record["genes"]:
        model.genes.get_by_id(gene).knock_out()

    mapped_background, missing_background = [], []
    for reaction_id in record["background_ids"]:
        resolved = resolve_reaction_id(model, reaction_id, aliases)
        if resolved is None:
            missing_background.append(reaction_id)
        else:
            model.reactions.get_by_id(resolved).lower_bound = UPTAKE_LOWER_BOUND
            mapped_background.append(resolved)

    ko_status, ko_growth = solve_growth(model)

    mapped_rescue, missing_rescue = [], []
    for reaction_id in record["rescue_ids"]:
        resolved = resolve_reaction_id(model, reaction_id, aliases)
        if resolved is None:
            missing_rescue.append(reaction_id)
        else:
            model.reactions.get_by_id(resolved).lower_bound = UPTAKE_LOWER_BOUND
            mapped_rescue.append(resolved)

    rescue_status, rescue_growth = solve_growth(model)
    classification = classify_auxotrophy(
        ko_status, ko_growth, rescue_status, rescue_growth, threshold
    )

    return {
        "pair_id": record["pair_id"], "excel_row": record["excel_row"],
        "gene_field": record["gene_field"], "chemical": record["chemical"],
        "n_genes": record["n_genes"],
        "strain_background": record["strain_background"],
        "ko_status": ko_status, "ko_growth": ko_growth,
        "rescue_status": rescue_status, "rescue_growth": rescue_growth,
        "classification": classification, "correct": classification == "correct",
        "missing_genes": "",
        "mapped_background": "+".join(mapped_background),
        "missing_background": "+".join(missing_background),
        "mapped_rescue": "+".join(mapped_rescue),
        "missing_rescue": "+".join(missing_rescue),
    }

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()

def format_duration(seconds):
    seconds = max(0, int(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}h {minutes:02d}m {seconds:02d}s" if hours else f"{minutes:02d}m {seconds:02d}s"

def benchmark_signature(model_path, model_label, threshold):
    payload = {
        "workflow_version": "0.5.2",
        "model": model_label,
        "model_sha256": sha256_file(model_path),
        "dataset_sha256": sha256_file(DATASET_XLSX),
        "solver": SOLVER,
        "threshold": threshold,
        "uptake_lower_bound": UPTAKE_LOWER_BOUND,
        "mode": ISOLATION_MODE,
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()

def run_benchmark(base_model, model_path, model_label, pairs_df, threshold, aliases, resume=True):
    checkpoint_csv = RESULTS_DIR / f"02_checkpoint_{model_label}_{ISOLATION_MODE}.csv"
    checkpoint_json = RESULTS_DIR / f"02_checkpoint_{model_label}_{ISOLATION_MODE}.json"
    signature = benchmark_signature(model_path, model_label, threshold)

    completed = pd.DataFrame()
    completed_ids = set()
    if resume and checkpoint_csv.exists() and checkpoint_json.exists():
        metadata = json.loads(checkpoint_json.read_text(encoding="utf-8"))
        if metadata.get("signature") != signature:
            raise RuntimeError(f"Checkpoint protocol mismatch for {model_label}.")
        completed = pd.read_csv(checkpoint_csv)
        completed_ids = set(completed["pair_id"].astype(str))

    new_results = []
    recent_times = []
    session_start = time.perf_counter()
    total = len(pairs_df)

    if completed_ids:
        print(f"{model_label}: resuming from {len(completed_ids)}/{total} completed pairs.", flush=True)
    else:
        print(f"{model_label}: starting benchmark with {total} pairs.", flush=True)

    for record in pairs_df.to_dict("records"):
        if record["pair_id"] in completed_ids:
            continue

        already_done = len(completed) + len(new_results)
        print(
            f"START {already_done + 1:3d}/{total} | {record['pair_id']} | "
            f"{record['gene_field']} | {record['chemical']}",
            flush=True,
        )

        pair_start = time.perf_counter()
        model = base_model.copy()
        result = simulate_pair(model, record, threshold, aliases)
        new_results.append(result)
        pair_seconds = time.perf_counter() - pair_start

        current = pd.concat([completed, pd.DataFrame(new_results)], ignore_index=True)
        current = (
            current.drop_duplicates(subset=["pair_id"], keep="last")
            .sort_values("excel_row")
            .reset_index(drop=True)
        )
        current.to_csv(checkpoint_csv, index=False)
        checkpoint_json.write_text(
            json.dumps(
                {
                    "signature": signature,
                    "model": model_label,
                    "isolation_mode": ISOLATION_MODE,
                },
                indent=2,
            ),
            encoding="utf-8",
        )

        recent_times.append(pair_seconds)
        recent_times = recent_times[-10:]
        done = len(current)
        remaining = total - done
        eta = float(np.mean(recent_times)) * remaining
        elapsed = time.perf_counter() - session_start

        del model
        if done % 10 == 0:
            gc.collect()

        print(
            f"DONE  {done:3d}/{total} | {record['pair_id']} | "
            f"class={result['classification']} | "
            f"KO={result['ko_status']}:{result['ko_growth']:.6g} | "
            f"rescue={result['rescue_status']}:{result['rescue_growth']:.6g} | "
            f"last={format_duration(pair_seconds)} | "
            f"elapsed={format_duration(elapsed)} | ETA={format_duration(eta)}",
            flush=True,
        )

    final = pd.concat([completed, pd.DataFrame(new_results)], ignore_index=True)
    final = (
        final.drop_duplicates(subset=["pair_id"], keep="last")
        .sort_values("excel_row")
        .reset_index(drop=True)
    )
    final.to_csv(RESULTS_DIR / f"02_{model_label}_pair_results.csv", index=False)
    print(f"{model_label}: DONE ({len(final)}/{total})", flush=True)
    return final


In [ ]:
original_base_model = read_sbml_model(str(ORIGINAL_XML))
original_base_model.solver = SOLVER

wt_growth = original_base_model.slim_optimize(error_value=np.nan)
if str(original_base_model.solver.status).lower() != "optimal":
    raise RuntimeError("Cannot calculate the viability threshold from an optimal WT solution.")

viability_threshold = VIABILITY_FRACTION * float(wt_growth)

print("Original model loaded once for the benchmark.")
print("WT growth:", wt_growth)
print("Viability threshold:", viability_threshold)


In [ ]:
original_results = run_benchmark(
    original_base_model,
    ORIGINAL_XML,
    "Yeast9",
    pairs,
    viability_threshold,
    ALIASES["Yeast9"],
    resume=True,
)


In [ ]:
curated_base_model = read_sbml_model(str(CURATED_XML))
curated_base_model.solver = SOLVER

curated_results = run_benchmark(
    curated_base_model,
    CURATED_XML,
    "Yeast9_curated",
    pairs,
    viability_threshold,
    ALIASES["Yeast9_curated"],
    resume=True,
)


In [ ]:
def summarize(results):
    return {
        "n": len(results),
        "correct": int((results["classification"] == "correct").sum()),
        "type_I": int((results["classification"] == "type_I").sum()),
        "type_II": int((results["classification"] == "type_II").sum()),
        "solver_error": int((results["classification"] == "solver_error").sum()),
        "input_error": int((results["classification"] == "input_error").sum()),
    }

summary = pd.DataFrame([
    {"model": "Yeast9", **summarize(original_results)},
    {"model": "Yeast9_curated", **summarize(curated_results)},
])
summary["accuracy"] = summary["correct"] / summary["n"]
display(summary)

comparison = original_results.merge(
    curated_results,
    on="pair_id",
    suffixes=("_original", "_curated"),
    validate="one_to_one",
)
comparison["change"] = np.select(
    [
        (comparison["classification_original"] != "correct") & (comparison["classification_curated"] == "correct"),
        (comparison["classification_original"] == "correct") & (comparison["classification_curated"] != "correct"),
    ],
    ["fixed", "regression"],
    default="unchanged",
)

display(comparison["change"].value_counts().rename_axis("change").to_frame("n"))
summary.to_csv(RESULTS_DIR / "02_benchmark_summary.csv", index=False)
comparison.to_csv(RESULTS_DIR / "02_pairwise_comparison.csv", index=False)

## Interpretation checkpoint

Do not interpret accuracy if `solver_error` or `input_error` is non-zero. The paper reports 93/147 correct predictions for Yeast9 and 117/147 for the curated model, but those values are comparison targets, not forced expected outputs.